In [ ]:
from pathlib import Path
import sys
import os


PROJECT_ROOT = Path.cwd()

# If the notebook is executed from /notebooks
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))


print("Project root:", PROJECT_ROOT)
print("Source root:", SRC_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
from deap_fusion.config import (
    LABEL_TYPE,
    MAX_SUBJECT_ID,
    NUM_FRAMES,
    BATCH_SIZE,
    WEIGHT_DECAY,
    EPOCHS,
    LR,
    FREEZE_BACKBONE,
    TRAIN_RATIO,
    SEED,
    NUM_WORKERS,
)

from deap_fusion.training.common import set_seeds

from deap_fusion.experiments.main import (
    run_main_experiment,
)

from deap_fusion.experiments.cv import (
    run_Kfold_cv_all_models,
)
# =========================================================
# 3. DATASET SOURCE
# =========================================================


from deap_fusion.data.download import data_download


# ---------------------------------------------------------
# Choose how the datasets are provided
#
# "local" -> datasets already exist under ./data/
# "drive" -> download private ZIP files using Colab Secrets
# ---------------------------------------------------------
DATA_SOURCE = "local"

In [ ]:
# =========================================================
# 3.1 DATASET PATHS
# =========================================================

if DATA_SOURCE == "local":

    DEAP_ROOT = PROJECT_ROOT / "data" / "eeg"
    IMAGE_ROOT = PROJECT_ROOT / "data" / "face_crops"


elif DATA_SOURCE == "drive":

    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError(
            "DATA_SOURCE='drive' is intended for Google Colab. "
            "Use DATA_SOURCE='local' outside Colab."
        )

    # Private Google Drive file IDs stored in Colab Secrets.
    eeg_file_id = userdata.get("DEAP_EEG_DRIVE_ID")
    face_file_id = userdata.get("DEAP_FACE_DRIVE_ID")

    if not eeg_file_id:
        raise ValueError(
            "Missing Colab Secret: DEAP_EEG_DRIVE_ID"
        )

    if not face_file_id:
        raise ValueError(
            "Missing Colab Secret: DEAP_FACE_DRIVE_ID"
        )

    # data_download() is idempotent:
    # if the folder already contains data, it won't download again.
    DEAP_ROOT = data_download(
        eeg_file_id,
        "eeg",
    )

    IMAGE_ROOT = data_download(
        face_file_id,
        "face_crops",
    )


else:
    raise ValueError(
        "DATA_SOURCE must be either 'local' or 'drive'."
    )


print("EEG root:", DEAP_ROOT)
print("Face crops root:", IMAGE_ROOT)

In [ ]:
# =========================================================
# 3.2 DATASET SANITY CHECK
# =========================================================

eeg_dat_dir = DEAP_ROOT / "data_preprocessed_python"

if not eeg_dat_dir.exists():
    raise FileNotFoundError(
        f"EEG folder not found: {eeg_dat_dir}"
    )

eeg_files = sorted(eeg_dat_dir.glob("*.dat"))

if len(eeg_files) == 0:
    raise RuntimeError(
        f"No DEAP .dat files found under {eeg_dat_dir}"
    )

if not IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f"Face crops folder not found: {IMAGE_ROOT}"
    )

face_trial_dirs = [
    p for p in IMAGE_ROOT.iterdir()
    if p.is_dir()
]

print(f"EEG subjects found: {len(eeg_files)}")
print(f"Face trial folders found: {len(face_trial_dirs)}")
print("Dataset setup OK.")

In [ ]:
# =========================================================
# 4. EXPERIMENT CONFIGURATION
# =========================================================

TARGET_NAME = "valence" if LABEL_TYPE == 0 else "arousal"

print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)

print(f"Target:               {TARGET_NAME}")
print(f"Label type:           {LABEL_TYPE}")
print(f"Max subject id:       {MAX_SUBJECT_ID}")
print(f"Video frames/trial:   {NUM_FRAMES}")
print(f"Batch size:           {BATCH_SIZE}")
print(f"EEG epochs:           {EPOCHS}")
print(f"Learning rate:        {LR}")
print(f"Weight decay:         {WEIGHT_DECAY}")
print(f"Train ratio:          {TRAIN_RATIO}")
print(f"Seed:                 {SEED}")
print(f"Workers:              {NUM_WORKERS}")
print(f"Freeze video backbone:{FREEZE_BACKBONE}")

In [ ]:
# =========================================================
# 5. REPRODUCIBILITY
# =========================================================

set_seeds(SEED)

print(f"Random seed initialized to {SEED}")

In [ ]:
# =========================================================
# 6. SHARED SPLIT SANITY CHECK
# =========================================================

from deap_fusion.data.splits import (
    init_shared_splits,
    get_shared_split,
)

init_shared_splits(
    eeg_root=DEAP_ROOT,
    image_root=IMAGE_ROOT,
    label_type=LABEL_TYPE,
    max_subject_id=MAX_SUBJECT_ID,
    split_mode="subject_dependent",
    train_ratio=TRAIN_RATIO,
    seed=SEED,
)

example_sid = "s01"

train_pairs, val_pairs, split_info = get_shared_split(
    sid=example_sid,
    split_mode="subject_dependent",
)

print(f"Subject: {example_sid}")
print("Split info:")
print(split_info)

print()
print("Train trials:", len(train_pairs))
print("Validation trials:", len(val_pairs))

print()
print("First train pairs:", train_pairs[:5])
print("First validation pairs:", val_pairs[:5])

In [ ]:
# =========================================================
# 7. MAIN EXPERIMENT - SINGLE 75/25 SHARED SPLIT
# =========================================================

from deap_fusion.experiments.main import run_main_experiment

main_results = run_main_experiment(
    eeg_root=DEAP_ROOT,
    image_root=IMAGE_ROOT,
    label_type=LABEL_TYPE,
    seed=SEED,
)

all_eeg_sd = main_results["EEG"]
all_video_sd = main_results["VIDEO"]
all_fusion_sd = main_results["FUSION"]
all_concat_fusion_sd = main_results["CONCAT"]
all_decision_fusion_sd = main_results["DECISION"]

In [ ]:
# =========================================================
# 8. MAIN EXPERIMENT QUICK CHECK
# =========================================================

for model_name, result in main_results.items():

    if result is None:
        print(f"{model_name}: missing")
        continue

    subject_results = result.get("subject_results", {})

    print(
        f"{model_name}: "
        f"{len(subject_results)} subject results"
    )

In [ ]:
# =========================================================
# 9. 4-FOLD CROSS-VALIDATION
# =========================================================

cv_results = run_Kfold_cv_all_models(
    eeg_root=DEAP_ROOT,
    image_root=IMAGE_ROOT,

    label_type=LABEL_TYPE,
    split_mode="subject_dependent",
    n_splits=4,
    max_subject_id=MAX_SUBJECT_ID,
    seed=SEED,

    run_eeg=True,
    run_video=True,
    run_film=True,
    run_concat=True,
    run_decision=True,

    eeg_mode=2,
    lambda_cons=0.0,

    eeg_epochs=EPOCHS,
    video_epochs=EPOCHS,

    fusion_stage1_epochs=15,
    fusion_stage2_epochs=5,

    batch_size=BATCH_SIZE,
)

In [ ]:
# =========================================================
# 10. CV QUICK CHECK
# =========================================================

print("Number of completed folds:", len(cv_results))

for fold, fold_results in cv_results.items():

    print("\n" + "-" * 60)
    print(f"Fold {fold}")
    print("-" * 60)

    for model_name in [
        "EEG",
        "VIDEO",
        "FUSION",
        "CONCAT",
        "DECISION",
    ]:
        result = fold_results.get(model_name)

        if result is None:
            print(f"{model_name}: missing")
            continue

        subject_results = result.get(
            "subject_results",
            {}
        )

        print(
            f"{model_name}: "
            f"{len(subject_results)} subjects"
        )

In [ ]:
# =========================================================
# 11. RESULTS & VISUALIZATION IMPORTS
# =========================================================

from deap_fusion.evaluation.results import (
    collect_cv_results,
    build_cv_fold_table,
    summarize_cv_results,
    make_cv_paper_table,
    compare_cv_models,
    compare_two_cv_models,
    compare_all_metrics,
    compare_film_vs_concat_all_metrics,
    collect_cv_film_stats,
    build_cv_film_summary_table,
)

from deap_fusion.evaluation.plots import (
    plot_cv_summary_bar,
    plot_cv_fold_lines,
    plot_subject_heatmap,
    plot_cv_model_difference,
    plot_film_parameter_histograms,
    plot_film_boxplot_by_label,
    plot_film_boxplot_correct_vs_wrong,
)

In [ ]:
# =========================================================
# 12. BUILD CV RESULT TABLES
# =========================================================

cv_subject_df = collect_cv_results(cv_results)

cv_fold_df = build_cv_fold_table(cv_subject_df)

cv_summary_df = summarize_cv_results(cv_fold_df)

cv_paper_table = make_cv_paper_table(cv_summary_df)

In [ ]:
# =========================================================
# 13. MAIN CV TABLES
# =========================================================

print("Subject-level CV results")
display(cv_subject_df)

print("\nFold-level CV results")
display(cv_fold_df)

print("\nFinal CV summary")
display(cv_summary_df)

print("\nPaper-ready table")
display(cv_paper_table)

In [ ]:
# =========================================================
# 14. MODEL COMPARISONS
# =========================================================

cv_vs_video_df = compare_cv_models(
    cv_fold_df,
    metric="best_val_acc",
    baseline="VIDEO",
)

cv_vs_concat_df = compare_cv_models(
    cv_fold_df,
    metric="best_val_acc",
    baseline="CONCAT",
)

cv_vs_eeg_df = compare_cv_models(
    cv_fold_df,
    metric="best_val_acc",
    baseline="EEG",
)

film_vs_concat_df = compare_two_cv_models(
    cv_fold_df,
    model_a="FUSION",
    model_b="CONCAT",
    metric="best_val_acc",
)

cv_all_metric_comparisons_df = compare_all_metrics(
    cv_fold_df,
    baselines=("VIDEO", "CONCAT", "EEG"),
)

film_vs_concat_all_metrics_df = (
    compare_film_vs_concat_all_metrics(
        cv_fold_df
    )
)

In [ ]:
# =========================================================
# 15. DISPLAY MODEL COMPARISONS
# =========================================================

print("Comparison against VIDEO")
display(cv_vs_video_df)

print("\nComparison against CONCAT")
display(cv_vs_concat_df)

print("\nComparison against EEG")
display(cv_vs_eeg_df)

print("\nDirect comparison: FiLM vs Concat")
display(film_vs_concat_df)

print("\nAll metric comparisons")
display(cv_all_metric_comparisons_df)

print("\nFiLM vs Concat across all metrics")
display(film_vs_concat_all_metrics_df)

In [ ]:
# =========================================================
# 16. FiLM MODULATION ANALYSIS ACROSS CV
# =========================================================

cv_film_stats_df = collect_cv_film_stats(
    cv_results
)

cv_film_summary_df = build_cv_film_summary_table(
    cv_film_stats_df
)

print("FiLM gamma / beta / gate trial-level analysis")
display(cv_film_stats_df)

print("\nFiLM gamma / beta / gate summary")
display(cv_film_summary_df)

In [ ]:
# =========================================================
# 17. CV PERFORMANCE PLOTS
# =========================================================

# Final performance with error bars
plot_cv_summary_bar(
    cv_summary_df,
    metric="best_val_acc",
    title="4-fold cross-validation: best validation accuracy",
)

# Stability across folds
plot_cv_fold_lines(
    cv_fold_df,
    metric="best_val_acc",
    title="Model stability across 4 folds",
)

# FiLM vs Concat
plot_cv_model_difference(
    cv_fold_df,
    model_a="FUSION",
    model_b="CONCAT",
    metric="best_val_acc",
)

# FiLM subject heatmap
plot_subject_heatmap(
    cv_subject_df,
    model="FUSION",
    metric="best_val_acc",
    title="FiLM fusion performance per subject and fold",
)

# Concat subject heatmap
plot_subject_heatmap(
    cv_subject_df,
    model="CONCAT",
    metric="best_val_acc",
    title="Concat fusion performance per subject and fold",
)

In [ ]:
# =========================================================
# 18. FiLM MODULATION PLOTS
# =========================================================

plot_film_parameter_histograms(
    cv_film_stats_df
)

# Compare modulation according to class label
plot_film_boxplot_by_label(
    cv_film_stats_df,
    "gamma_deviation_from_1",
)

plot_film_boxplot_by_label(
    cv_film_stats_df,
    "beta_abs_mean",
)

plot_film_boxplot_by_label(
    cv_film_stats_df,
    "gate_mean",
)

plot_film_boxplot_by_label(
    cv_film_stats_df,
    "relative_modulation",
)

# Compare correct vs incorrect predictions
plot_film_boxplot_correct_vs_wrong(
    cv_film_stats_df,
    "relative_modulation",
)

plot_film_boxplot_correct_vs_wrong(
    cv_film_stats_df,
    "beta_abs_mean",
)

plot_film_boxplot_correct_vs_wrong(
    cv_film_stats_df,
    "gate_mean",
)

In [ ]:
# =========================================================
# 19. SAVE CV RESULTS
# =========================================================

from pathlib import Path

out_dir = PROJECT_ROOT / "outputs" / "metrics" / "cv"
out_dir.mkdir(parents=True, exist_ok=True)

cv_subject_df.to_csv(
    out_dir / f"cv4_subject_results_label{LABEL_TYPE}.csv",
    index=False,
)

cv_fold_df.to_csv(
    out_dir / f"cv4_fold_results_label{LABEL_TYPE}.csv",
    index=False,
)

cv_summary_df.to_csv(
    out_dir / f"cv4_summary_label{LABEL_TYPE}.csv",
    index=False,
)

cv_paper_table.to_csv(
    out_dir / f"cv4_paper_table_label{LABEL_TYPE}.csv",
    index=False,
)

cv_vs_video_df.to_csv(
    out_dir / f"cv4_comparison_vs_video_label{LABEL_TYPE}.csv",
    index=False,
)

cv_vs_concat_df.to_csv(
    out_dir / f"cv4_comparison_vs_concat_label{LABEL_TYPE}.csv",
    index=False,
)

cv_vs_eeg_df.to_csv(
    out_dir / f"cv4_comparison_vs_eeg_label{LABEL_TYPE}.csv",
    index=False,
)

film_vs_concat_df.to_csv(
    out_dir / f"cv4_film_vs_concat_label{LABEL_TYPE}.csv",
    index=False,
)

cv_all_metric_comparisons_df.to_csv(
    out_dir / f"cv4_all_metric_comparisons_label{LABEL_TYPE}.csv",
    index=False,
)

film_vs_concat_all_metrics_df.to_csv(
    out_dir / f"cv4_film_vs_concat_all_metrics_label{LABEL_TYPE}.csv",
    index=False,
)

cv_film_stats_df.to_csv(
    out_dir / f"cv4_film_stats_label{LABEL_TYPE}.csv",
    index=False,
)

cv_film_summary_df.to_csv(
    out_dir / f"cv4_film_summary_label{LABEL_TYPE}.csv",
    index=False,
)

print("Saved CV results to:", out_dir)